In [2]:
from transformers import pipeline
import json
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch
from datasets import Dataset
import numpy as np
from nltk.tokenize import word_tokenize
import os
from sklearn.model_selection import train_test_split

In [3]:
device = "cuda"

Загрузка датасета в очищенном формате и модели

In [4]:
with open("../data/formatted_data.json", "r") as f:
    data = json.load(f)

In [5]:
data[:3]

[{'text': "I have a child. I feel like at his age it's time to brush his teeth. But I don't know if it's too early to start. Also, I don't know which toothpaste to choose, with or without fluoride. I'm also wondering what kind of toothbrush would be safe for a young child.",
  'keywords': ['child', 'fluoride', 'toothpaste', 'toothbrush']},
 {'text': 'I have high blood pressure and diabetes. But I like to drink coffee. How many cups of coffee can I drink a day? When is it better for me to drink it during the day? What type of coffee  should I drink?',
  'keywords': ['high blood pressure',
   'diabetes',
   'coffee consumption',
   'coffee type']},
 {'text': 'I have an infant. I want to make a crib for him to sleep in. I heard that babies that age should sleep without a blanket and pillow? Is that true?',
  'keywords': ['infant', 'sleep', 'crib', 'blanket', 'pillow']}]

Разделение на train и test выборки

In [6]:
train_data, eval_data = train_test_split(data, test_size=0.2, shuffle=True, random_state=42)

Загрузка модели и токенизатора

In [7]:
model_name = "ilsilfverskiold/bart-keyword-extractor"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

In [8]:
def preprocess_function(examples):
    inputs = [text for text in examples['text']]
    targets = [', '.join(keywords) for keywords in examples['keywords']]
    
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')
    
    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

In [9]:
train_dataset = Dataset.from_dict({
    'text': [x['text'] for x in train_data],
    'keywords': [x['keywords'] for x in train_data]
}).map(preprocess_function, batched=True)

eval_dataset = Dataset.from_dict({
    'text': [x['text'] for x in eval_data],
    'keywords': [x['keywords'] for x in eval_data]
}).map(preprocess_function, batched=True)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [10]:
output_dir = "./results"
logging_dir = "./logs"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(logging_dir, exist_ok=True)

In [11]:
for path in [output_dir, logging_dir]:
    if os.path.exists(path) and not os.path.isdir(path):
        os.remove(path)
    os.makedirs(path, exist_ok=True)

In [12]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True if torch.cuda.is_available() else False,
    lr_scheduler_type="linear",
    warmup_steps=500,
    seed=42,
    optim="adamw_torch",
    logging_dir=logging_dir,
    logging_steps=100,
)

In [13]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    pred_keywords = [set(word_tokenize(pred.lower())) for pred in decoded_preds]
    true_keywords = [set(word_tokenize(label.lower())) for label in decoded_labels]
    

    precisions, recalls, f1s = [], [], []
    for pred, true in zip(pred_keywords, true_keywords):
        if len(pred) == 0 or len(true) == 0:
            continue
            
        common = pred & true
        precision = len(common) / len(pred) if len(pred) > 0 else 0
        recall = len(common) / len(true) if len(true) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    
    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    avg_f1 = np.mean(f1s)
    
    return {
        'precision': avg_precision,
        'recall': avg_recall,
        'f1': avg_f1
    }

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,16.702669,0.813889,0.877778,0.840067
2,No log,16.702669,0.813889,0.877778,0.840067
3,No log,16.675722,0.813889,0.877778,0.840067


c:\Users\1379\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=3, training_loss=3.0984233220418296, metrics={'train_runtime': 139.2144, 'train_samples_per_second': 0.517, 'train_steps_per_second': 0.022, 'total_flos': 78015765676032.0, 'train_loss': 3.0984233220418296, 'epoch': 3.0})

In [14]:
results = trainer.evaluate()
print(results)

{'eval_loss': 16.675722122192383, 'eval_precision': 0.813888888888889, 'eval_recall': 0.8777777777777778, 'eval_f1': 0.8400673400673401, 'eval_runtime': 7.8175, 'eval_samples_per_second': 0.768, 'eval_steps_per_second': 0.128, 'epoch': 3.0}


In [15]:
def extract_keywords(text, model, tokenizer, max_length=128):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    output = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        num_beams=5,
        early_stopping=True
    ).to(device)
    
    keywords = tokenizer.decode(output[0], skip_special_tokens=True)
    return keywords.split(', ')

text = "Who is at risk for Pericarditis?"
keywords = extract_keywords(text, model, tokenizer)
print(keywords)

['Pericarditis', 'risk', 'individuals']


In [16]:
model.save_pretrained("./medical_keyword_extractor")
tokenizer.save_pretrained("./medical_keyword_extractor")

('./medical_keyword_extractor\\tokenizer_config.json',
 './medical_keyword_extractor\\special_tokens_map.json',
 './medical_keyword_extractor\\vocab.json',
 './medical_keyword_extractor\\merges.txt',
 './medical_keyword_extractor\\added_tokens.json')

In [17]:
model = BartForConditionalGeneration.from_pretrained("./medical_keyword_extractor")
tokenizer = BartTokenizer.from_pretrained("./medical_keyword_extractor")